In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
import re
import random
import math
import time
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"{device}")

cuda


In [12]:
# Data cleaning and reading
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zа-яё\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load_data(filepath, num_examples=30000):
    eng_sentences = []
    rus_sentences = []

    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[:num_examples]:
            parts = line.split('\t')
            if len(parts) >= 2:
                eng_sentences.append(clean_text(parts[0]))
                rus_sentences.append(clean_text(parts[1]))

    return eng_sentences, rus_sentences

eng_data, rus_data = load_data('../../rus.txt', 100000)

print(f"Count of sentences: {len(eng_data)}")
print("Example (Eng):", eng_data[100])
print("Example (Rus):", rus_data[100])

Count of sentences: 100000
Example (Eng): he ran
Example (Rus): он бежал


In [13]:
# Vocabulary
class Vocabulary:
    def __init__(self, name):
        self.name = name
        self.word2idx = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
        self.idx2word = {0: '<pad>', 1: '<sos>', 2: '<eos>', 3: '<unk>'}
        self.word_freq = Counter()
        self.idx = 4

    def add_sentence(self, sentence):
        for word in sentence.split():
            self.word_freq[word] += 1
            if word not in self.word2idx:
                self.word2idx[word] = self.idx
                self.idx2word[self.idx] = word
                self.idx += 1

    def __len__(self):
        return len(self.word2idx)

# creating vocabs
eng_vocab = Vocabulary("English")
rus_vocab = Vocabulary("Russian")

for eng, rus in zip(eng_data, rus_data):
    eng_vocab.add_sentence(eng)
    rus_vocab.add_sentence(rus)

print(f"Length of english vocab: {len(eng_vocab)}")
print(f"Length of russian vocab: {len(rus_vocab)}")

Length of english vocab: 7216
Length of russian vocab: 20764


In [14]:
# Dataset and Collate function
class TranslationDataset(Dataset):
    def __init__(self, eng_sentences, rus_sentences, eng_vocab, rus_vocab):
        self.eng_sentences = eng_sentences
        self.rus_sentences = rus_sentences
        self.eng_vocab = eng_vocab
        self.rus_vocab = rus_vocab

    def __len__(self):
        return len(self.eng_sentences)

    def __getitem__(self, i):
        eng_sentence = self.eng_sentences[i]
        rus_sentence = self.rus_sentences[i]

        eng_tensor = [self.eng_vocab.word2idx.get(w, 3) for w in eng_sentence.split()]

        # Adding <sos> (1) and <eos> (2)
        rus_tensor = [1] + [self.rus_vocab.word2idx.get(w, 3) for w in rus_sentence.split()] + [2]

        return torch.tensor(eng_tensor), torch.tensor(rus_tensor)

def collate_fn(batch):
    eng_batch, rus_batch = [], []
    for eng_item, rus_item in batch:
        eng_batch.append(eng_item)
        rus_batch.append(rus_item)

    # Padding with 0s
    eng_batch = pad_sequence(eng_batch, padding_value=0, batch_first=False)
    rus_batch = pad_sequence(rus_batch, padding_value=0, batch_first=False)

    return eng_batch, rus_batch

# Data splitting (80% train, 20% validation)
eng_train, eng_val, rus_train, rus_val = train_test_split(eng_data, rus_data, test_size=0.2, random_state=42)

train_dataset = TranslationDataset(eng_train, rus_train, eng_vocab, rus_vocab)
val_dataset = TranslationDataset(eng_val, rus_val, eng_vocab, rus_vocab)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn)

for src, trg in train_loader:
    print("Length of Source (English) tensor (SeqLen, BatchSize):", src.shape)
    print("Length of target (Russian) tensor (SeqLen, BatchSize):", trg.shape)
    break

Length of Source (English) tensor (SeqLen, BatchSize): torch.Size([6, 128])
Length of target (Russian) tensor (SeqLen, BatchSize): torch.Size([8, 128])


In [15]:
# Model Architecture
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, dropout_rate=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.droput = nn.Dropout(dropout_rate)
        self.lstm = nn.LSTM(emb_dim, hid_dim, num_layers=2, dropout=dropout_rate)

    def forward(self, src):
        # src: [seq_len, batch_size]
        embedded = self.droput(self.embedding(src))
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, dropout_rate=0.5):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.dropout = nn.Dropout(dropout_rate)
        self.lstm = nn.LSTM(emb_dim, hid_dim, num_layers=2, dropout=dropout_rate)
        self.fc_out = nn.Linear(hid_dim, output_dim)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0)) # [batch_size, output_dim]
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src = [src_len, batch_size]
        # trg = [trg_len, batch_size]
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        hidden, cell = self.encoder(src)

        # First input token is <sos>
        input = trg[0, :]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output

            top1 = output.argmax(1)

            teacher_force = random.random() < teacher_forcing_ratio
            input = trg[t] if teacher_force else top1

        return outputs

In [16]:
# Hyperparameters and model initialization
INPUT_DIM = len(eng_vocab)
OUTPUT_DIM = len(rus_vocab)
EMB_DIM = 256
HID_DIM = 1024
ENC_DROPOUT = 0.2
DEC_DROPOUT = 0.2

enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

PAD_IDX = rus_vocab.word2idx['<pad>']
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# Adam optimizer (lr=0.001)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters:')

The model has 55,741,724 trainable parameters:


In [17]:
# Training and Evaluation
def train(model, iterator, optimizer, criterion, clip, teacher_forcing_ratio):
    model.train()
    epoch_loss = 0

    for i, (src, trg) in enumerate(iterator):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()

        output = model(src, trg, teacher_forcing_ratio=teacher_forcing_ratio)

        # output: [trg_len, batch_size, output_dim] ->  [ (trg_len-1)*batch_size, output_dim ]
        # trg: [trg_len, batch_size] -> [ (trg_len-1)*batch_size ]
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)

        loss = criterion(output, trg)
        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for i, (src, trg) in enumerate(iterator):
            src, trg = src.to(device), trg.to(device)

            # teacher_forcing_ratio = 0
            output = model(src, trg, teacher_forcing_ratio=0)

            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [18]:
# Training loop
N_EPOCHS = 30
CLIP = 1.0

best_valid_loss = float('inf')

# --- Early Stopping parameters ---
patience = 5
epochs_no_improve = 0

print("Starting training...\n")

for epoch in range(N_EPOCHS):
    start_time = time.time()

    current_teacher_forcing = max(0.2, 1.0 - (epoch * 0.1))

    train_loss = train(model, train_loader, optimizer, criterion, CLIP, current_teacher_forcing)
    valid_loss = evaluate(model, val_loader, criterion)

    end_time = time.time()

    print(f'Epoch: {epoch+1:02} | Time: {end_time - start_time:.1f}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), '../../seq2seq_best_model.pt')
        epochs_no_improve = 0
        print("\t Saved new best model:")
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patience:
        print(f"\n[!!!] Early Stopping on {epoch+1} epoch!")
        break

Starting training...

Epoch: 01 | Time: 120.8s
	Train Loss: 4.035 | Train PPL:  56.533
	 Val. Loss: 3.764 |  Val. PPL:  43.111
	 Saved new best model:
Epoch: 02 | Time: 117.2s
	Train Loss: 2.405 | Train PPL:  11.078
	 Val. Loss: 2.719 |  Val. PPL:  15.169
	 Saved new best model:
Epoch: 03 | Time: 117.7s
	Train Loss: 1.653 | Train PPL:   5.222
	 Val. Loss: 2.313 |  Val. PPL:  10.109
	 Saved new best model:
Epoch: 04 | Time: 117.1s
	Train Loss: 1.275 | Train PPL:   3.579
	 Val. Loss: 2.150 |  Val. PPL:   8.587
	 Saved new best model:
Epoch: 05 | Time: 118.5s
	Train Loss: 1.077 | Train PPL:   2.936
	 Val. Loss: 2.072 |  Val. PPL:   7.938
	 Saved new best model:
Epoch: 06 | Time: 118.8s
	Train Loss: 0.979 | Train PPL:   2.663
	 Val. Loss: 2.012 |  Val. PPL:   7.480
	 Saved new best model:
Epoch: 07 | Time: 116.9s
	Train Loss: 0.919 | Train PPL:   2.507
	 Val. Loss: 1.986 |  Val. PPL:   7.286
	 Saved new best model:
Epoch: 08 | Time: 118.7s
	Train Loss: 0.883 | Train PPL:   2.417
	 Val. Los

In [20]:
# Inference
def translate_sentence(sentence, eng_vocab, rus_vocab, model, device, max_len=50):
    model.eval()

    if isinstance(sentence, str):
        sentence = clean_text(sentence)
        tokens = sentence.split()
    else:
        tokens = sentence

    text_to_indices = [eng_vocab.word2idx.get(token, eng_vocab.word2idx['<unk>']) for token in tokens]

    sentence_tensor = torch.LongTensor(text_to_indices).unsqueeze(1).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(sentence_tensor)

    outputs = [rus_vocab.word2idx['<sos>']]

    for _ in range(max_len):
        previous_word = torch.LongTensor([outputs[-1]]).to(device)

        with torch.no_grad():
            output, hidden, cell = model.decoder(previous_word, hidden, cell)
            best_guess = output.argmax(1).item()

        outputs.append(best_guess)

        if best_guess == rus_vocab.word2idx['<eos>']:
            break

    translated_sentence = [rus_vocab.idx2word[idx] for idx in outputs]

    return " ".join(translated_sentence[1:-1])

model.load_state_dict(torch.load('../../seq2seq_best_model.pt', weights_only=True))

test_sentences = [
    "i am cold",
    "he is a good boy",
    "how are you",
    "run away"
]

print("--- Testing Translations ---")
for sentence in test_sentences:
    translation = translate_sentence(sentence, eng_vocab, rus_vocab, model, device)
    print(f"English: {sentence}")
    print(f"Russian translation: {translation}")
    print("-" * 30)

--- Testing Translations ---
English: i am cold
Russian translation: я холодно
------------------------------
English: he is a good boy
Russian translation: он хороший мальчик
------------------------------
English: how are you
Russian translation: как вы
------------------------------
English: run away
Russian translation: бегите
------------------------------
